In [2]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path


sys.path.insert(0, str(Path.cwd().parent))

from src.config import settings
from src.database import SpotifyDB
from src.spotify import Spotify

In [3]:
db = SpotifyDB(db_path=settings.database_url)
stream_df = db.query("""
    SELECT *
    FROM streams
    ORDER BY [timestamp]
    LIMIT 1000;
""").df()
stream_df.head()

,timestamp,platform,ms_played,country,ip_address,track_name,artist_name,album_name,track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2016-11-24 04:57:00,"iOS 10.1.1 (iPhone8,1)",219753,US,148.75.19.184,Reptilia,The Strokes,Room On Fire,spotify:track:57Xjny5yNzAcsxnusKmAfA,None,...,None,None,None,clickrow,trackdone,False,False,False,<NA>,False
1,2016-11-24 04:57:09,"iOS 10.1.1 (iPhone8,1)",7301,US,148.75.19.184,No Cities To Love,Sleater-Kinney,No Cities to Love,spotify:track:0ZdviWQ9dmO2doDba1DtA2,None,...,None,None,None,trackdone,fwdbtn,False,False,False,<NA>,False
2,2016-11-24 04:57:11,"iOS 10.1.1 (iPhone8,1)",1996,US,148.75.19.184,R U Mine?,Arctic Monkeys,AM,spotify:track:2LlvrdnLa3XbB1b4jYuCnl,None,...,None,None,None,fwdbtn,fwdbtn,False,False,False,<NA>,False
3,2016-11-24 04:57:12,"iOS 10.1.1 (iPhone8,1)",1253,US,148.75.19.184,Take Me Out,Franz Ferdinand,Franz Ferdinand,spotify:track:6ooluO7DiEhI1zmK94nRCM,None,...,None,None,None,fwdbtn,backbtn,False,False,False,<NA>,False
4,2016-11-24 04:57:16,"iOS 10.1.1 (iPhone8,1)",3506,US,148.75.19.184,R U Mine?,Arctic Monkeys,AM,spotify:track:2LlvrdnLa3XbB1b4jYuCnl,None,...,None,None,None,backbtn,endplay,False,False,False,<NA>,False


Let's create some easy time features such as year, month, day, is_weekend, etc.

In [4]:
stream_df.track_uri.unique()

<StringArray>
['spotify:track:57Xjny5yNzAcsxnusKmAfA',
 'spotify:track:0ZdviWQ9dmO2doDba1DtA2',
 'spotify:track:2LlvrdnLa3XbB1b4jYuCnl',
 'spotify:track:6ooluO7DiEhI1zmK94nRCM',
 'spotify:track:6a6EnneRMZqlKW9NvOSUBk',
 'spotify:track:3V8sM5OOG6YfDuDLa2IIYJ',
 'spotify:track:3iMJAcu14R5hkiia91DWaN',
 'spotify:track:1EyHgX2x0w22h7Wl7aTVW1',
 'spotify:track:26oF6WjkOjDIRK9YsdZp2l',
 'spotify:track:6hVGw7yVUd2bpdHMPCWrtN',
 ...
 'spotify:track:3mLUGofMqtzGh2fqEuhjje',
 'spotify:track:2mSc1qAsAhcpPMWSWQvU0h',
 'spotify:track:470GRgn0wzQWq5EBTqEQ9R',
 'spotify:track:6PX8IpFbf84tV4l1D9S0bh',
 'spotify:track:3uS2P07FDjhm1GlXKWKilM',
 'spotify:track:50NAYMCplbHg8JHtVQ9yfd',
 'spotify:track:6FBWEpr7Tpxi25SsPvTPX1',
 'spotify:track:3CtPzKmKDMoiXp8V0LxeTP',
 'spotify:track:2g9tHzbprsfvDTDX3KFsIC',
 'spotify:track:7wxXwQzCQscTLvJ5JwCjXn']
Length: 588, dtype: str

In [5]:
month_to_season = {
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Autumn",
    10: "Autumn",
    11: "Autumn",
    12: "Winter",
}

In [6]:
data = stream_df.assign(
    year=stream_df["timestamp"].dt.year,
    month=stream_df["timestamp"].dt.month,
    month_name=stream_df["timestamp"].dt.month_name(),
    hour=stream_df["timestamp"].dt.hour,
    seconds=stream_df['timestamp'].dt.second,
    minute=stream_df["timestamp"].dt.minute,
    quarter="Q" + stream_df["timestamp"].dt.quarter.astype("string"),
    season=stream_df["timestamp"].dt.month.map(month_to_season),
    is_weekend=stream_df["timestamp"].dt.day_of_week > 4,  # Monday=0, Sunday=6
)

Features related to user behavior like:
- did_skip: if the song was played for less than 3 seconds (30,000ms)
  - Note: though Spotify provides a `skipped` column, the documentation does not state was criteria is used to flag this behvior
- completion_rate: how much of the song was listened to
- time_of_day: 

In [7]:
data["time_of_day"] = pd.cut(
    data["hour"],
    [0, 6, 12, 18, 24],
    right=False,
    labels=["Night", "Morning", "Afternoon", "Evening"],
)

In [8]:
data['did_skip'] = data['seconds'] < 3

In [9]:
sp = Spotify(client_id=settings.spotify_client_id, client_secret=settings.spotify_client_secret)

In [10]:
track_info = sp.get_track("11dFghVXANMlKmJXsNCbNl")

In [11]:
data.shape

(1000, 34)